In [1]:
import pandas as pd # it is used for data manipulation and analysis, providing data structures like DataFrame and Series
import numpy as np # it is used for data manipulation and analysis, providing data structures like DataFrame and Series
from sklearn.model_selection import train_test_split, GridSearchCV # it is used to split the dataset into training and testing sets, and to perform hyperparameter tuning using cross-validation
from sklearn.preprocessing import StandardScaler, OneHotEncoder # it is used to scale numerical features and encode categorical features
from sklearn.compose import ColumnTransformer # it is used to apply different preprocessing steps to different columns of the dataset
from sklearn.pipeline import Pipeline # it is used to create a pipeline that combines preprocessing and model training steps
from sklearn.linear_model import LogisticRegression # it is used to train a logistic regression model
from sklearn.ensemble import RandomForestClassifier # it is used to train a random forest classifier model
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score # it is used to evaluate the performance of the trained model using various metrics
import joblib # it is used to save and load the trained model to/from disk

In [2]:
df = pd.read_csv("train_cleaned.csv")

1. Split the data

In [3]:
x = df[["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked", "Family size"]]
y = df["Survived"]

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y) # it splits dataset into training and testing sets with 20% of data reserved for testing, using a random seed of 42 for reproducibility and stratifying by the target variable to maintain class distribution
print(x_train.shape, x_test.shape)

(712, 8) (179, 8)


2. Preprocessing pipeline

In [4]:
num_cols = ["Age", "SibSp", "Parch", "Fare", "Family size"]
cat_cols = ["Pclass", "Sex", "Embarked"]

preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_cols),       # scale numeric columns
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols) # one hot encode categorical columns
])

In [ ]:
preprocess  

,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,copy,True
,with_mean,True
,with_std,True


3. Baseline model: Logistic Regression

In [6]:
log_pipe = Pipeline([("prep", preprocess), ("model", LogisticRegression(max_iter=1000))])
log_pipe.fit(x_train, y_train)
print("Logistic Regression Test Accuracy: ", accuracy_score(y_test, log_pipe.predict(x_test)))

Logistic Regression Test Accuracy:  0.8044692737430168


4. A stronger model: Random Forest

In [7]:
rf_pipe = Pipeline([("prep", preprocess), ("model", RandomForestClassifier(n_estimators=200, random_state=42))])
rf_pipe.fit(x_train, y_train)
print("RandomForestClassifier Test Accuracy: ", accuracy_score(y_test, rf_pipe.predict(x_test)))


RandomForestClassifier Test Accuracy:  0.8379888268156425


5. Evaluate properly — not just accuracy

In [8]:
preds = rf_pipe.predict(x_test)
print(classification_report(y_test, preds))


              precision    recall  f1-score   support

           0       0.85      0.90      0.87       110
           1       0.82      0.74      0.78        69

    accuracy                           0.84       179
   macro avg       0.83      0.82      0.83       179
weighted avg       0.84      0.84      0.84       179



In [9]:
print(confusion_matrix(y_test, preds))

[[99 11]
 [18 51]]


6. Tune hyperparameters

In [10]:
param_grid = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [None, 10, 20],
}

grid = GridSearchCV(rf_pipe, param_grid, cv=5, scoring="accuracy")
grid.fit(x_train, y_train)

print("Best params: ", grid.best_params_)
print("Best CV score:", grid.best_score_)
print("Test accuracy with best model: ", accuracy_score(y_test, grid.predict(x_test)))

Best params:  {'model__max_depth': 10, 'model__n_estimators': 300}
Best CV score: 0.8119078104993598
Test accuracy with best model:  0.8156424581005587


7. Save and reload the model

In [11]:
joblib.dump(grid.best_estimator_, "titanic_model.joblib")

loaded = joblib.load("titanic_model.joblib")
same = np.allclose(grid.predict_proba(x_test), loaded.predict_proba(x_test))
print("Are the predictions from the original and loaded model the same? ", same)

Are the predictions from the original and loaded model the same?  True


8. Predict on a single new passenger

In [12]:
sample = x_test.iloc[0:1]
print("Prediction for sample: ", loaded.predict(sample))
print("Actual:", y_test.iloc[0])

Prediction for sample:  [0]
Actual: 0


In [13]:
print("Sample data:", sample)

Sample data:      Pclass   Sex  Age  SibSp  Parch   Fare Embarked  Family size
565       3  male   24      2      0  24.15        S            3


9. SVC

In [14]:
from sklearn.svm import SVC

In [15]:
svc_pipe = Pipeline([("prep", preprocess), ("model", SVC(probability=True))])

svc_pipe.fit(x_train, y_train)

svc_preds = svc_pipe.predict(x_test)

print("SVC accuracy: ", accuracy_score(y_test, svc_preds))

SVC accuracy:  0.8324022346368715


In [16]:
print(classification_report(y_test, svc_preds))

              precision    recall  f1-score   support

           0       0.83      0.91      0.87       110
           1       0.83      0.71      0.77        69

    accuracy                           0.83       179
   macro avg       0.83      0.81      0.82       179
weighted avg       0.83      0.83      0.83       179



9. Feature importance(For Random Forest pipeline)

In [19]:
feature_names = rf_pipe.named_steps["prep"].get_feature_names_out()

importances = rf_pipe.named_steps["model"].feature_importances_

feature_importances = pd.Series(importances, index=feature_names).sort_values(ascending=False)

print(feature_importances)


num__Fare           0.249253
num__Age            0.233847
cat__Sex_female     0.146042
cat__Sex_male       0.132041
cat__Pclass_3       0.052036
num__Family size    0.048201
cat__Pclass_1       0.029894
num__SibSp          0.027555
num__Parch          0.027467
cat__Pclass_2       0.016146
cat__Embarked_S     0.015516
cat__Embarked_C     0.012838
cat__Embarked_Q     0.009165
dtype: float64


    CONCLUSION